## Imports

In [ ]:
from pathlib import Path

import numpy as np
from PIL import Image, ImageDraw, ImageFont

import torch
from torchvision import transforms

import yolov5
from yolov5.models.yolo import Model
from yolov5.utils.general import check_yaml, yaml_load, non_max_suppression

## Util functions

In [ ]:
def letterbox_image_pil(img, new_size=640, color=(114, 114, 114)):
    img = img.convert('RGB')
    w0, h0 = img.size
    r = min(new_size / w0, new_size / h0)
    new_unpad = (int(round(w0 * r)), int(round(h0 * r)))
    dw = new_size - new_unpad[0]
    dh = new_size - new_unpad[1]
    dw /= 2
    dh /= 2

    img_resized = img.resize(new_unpad, Image.Resampling.BILINEAR)
    new_img = Image.new('RGB', (new_size, new_size), color)
    new_img.paste(img_resized, (int(dw), int(dh)))

    img_tensor = torch.from_numpy(np.array(new_img)).permute(2, 0, 1).float() / 255.
    return img_tensor.unsqueeze(0), r, (dw, dh), (w0, h0)

In [26]:
def scale_boxes_pil(input_shape, boxes, original_size):
    gain = min(input_shape[0] / original_size[1], input_shape[1] / original_size[0])
    pad_x = (input_shape[1] - original_size[0] * gain) / 2
    pad_y = (input_shape[0] - original_size[1] * gain) / 2

    boxes[:, [0, 2]] -= pad_x
    boxes[:, [1, 3]] -= pad_y
    boxes[:, :4] /= gain
    boxes[:, 0].clamp_(0, original_size[0])
    boxes[:, 1].clamp_(0, original_size[1])
    boxes[:, 2].clamp_(0, original_size[0])
    boxes[:, 3].clamp_(0, original_size[1])

    return boxes

In [ ]:
def scale_boxes_back(boxes, gain, pad, original_size):
    pad_w, pad_h = pad
    boxes[:, [0, 2]] -= pad_w
    boxes[:, [1, 3]] -= pad_h
    boxes[:, :4] /= gain

    boxes[:, [0, 2]] = boxes[:, [0, 2]].clamp(0, original_size[0])
    boxes[:, [1, 3]] = boxes[:, [1, 3]].clamp(0, original_size[1])
    return boxes

## Load model

In [16]:
# Load yolov5n.yaml from package
yolov5_dir = Path(yolov5.__file__).resolve().parent
model_cfg = yolov5_dir / 'models' / 'yolov5n.yaml'

In [17]:
# Build model
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.mps.is_available() else 'cpu')
data_yaml = check_yaml('../dataset/data.yaml')
data_dict = yaml_load(data_yaml)
nc = data_dict.get('nc', 1)

In [ ]:
# Load model structure + weights
model = Model(model_cfg, ch=3, nc=nc).to(device)
checkpoint = torch.load('yolov5n_custom.pt', map_location=device)
model.load_state_dict(checkpoint, strict=False)
model.eval()

Overriding model.yaml nc=80 with nc=6

                 from  n    params  module                                  arguments                     
  0                -1  1      1760  yolov5.models.common.Conv               [3, 16, 6, 2, 2]              
  1                -1  1      4672  yolov5.models.common.Conv               [16, 32, 3, 2]                
  2                -1  1      4800  yolov5.models.common.C3                 [32, 32, 1]                   
  3                -1  1     18560  yolov5.models.common.Conv               [32, 64, 3, 2]                
  4                -1  2     29184  yolov5.models.common.C3                 [64, 64, 2]                   
  5                -1  1     73984  yolov5.models.common.Conv               [64, 128, 3, 2]               
  6                -1  3    156928  yolov5.models.common.C3                 [128, 128, 3]                 
  7                -1  1    295424  yolov5.models.common.Conv               [128, 256, 3, 2]             

DetectionModel(
  (model): Sequential(
    (0): Conv(
      (conv): Conv2d(3, 16, kernel_size=(6, 6), stride=(2, 2), padding=(2, 2), bias=False)
      (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (1): Conv(
      (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (2): C3(
      (cv1): Conv(
        (conv): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (cv2): Conv(
        (conv): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
     

## Run model

In [28]:
img_path = '../data/cropped_images/image_157.png'

In [29]:
img_tensor, (w0, h0), img_letterboxed = load_image_pil(img_path)
img_tensor = img_tensor.to(device)

In [30]:
with torch.no_grad():
    preds = model(img_tensor)
    preds = non_max_suppression(preds[0], conf_thres=0.25, iou_thres=0.45)

In [32]:
original_image = Image.open(img_path).convert('RGB')
draw = ImageDraw.Draw(original_image)
font = ImageFont.load_default()
class_names = data_dict['names']

for det in preds:
    if len(det):
        det[:, :4] = scale_boxes_pil((640, 640), det[:, :4], original_image.size)
        for *xyxy, conf, cls in det:
            label = f"{class_names[int(cls)]} {conf:.2f}"
            draw.rectangle(xyxy, outline=(0, 255, 0), width=2)
            draw.text((xyxy[0], xyxy[1] - 10), label, fill=(255, 255, 255), font=font)

original_image.save('prediction_pil.jpg')
print("Saved PIL prediction as prediction_pil.jpg")

Saved PIL prediction as prediction_pil.jpg
